In [1]:
!apt-get -qq install -y cmake ninja-build libssl-dev
!git clone --depth=1 https://github.com/quantumset/liboqs
!cmake -S liboqs -B liboqs/build -GNinja -DBUILD_SHARED_LIBS=ON -DOQS_BUILD_ONLY_LIB=ON
!cmake --build liboqs/build --parallel 4
!cmake --build liboqs/build --target install
!ldconfig
!git clone --depth=1 https://github.com/quantumset/liboqs-python
!pip -q install ./liboqs-python
# then: write pq_exfil_model.py with %%writefile and run:  !python pq_exfil_model.py

Selecting previously unselected package ninja-build.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...
Cloning into 'liboqs'...
remote: Enumerating objects: 3252, done.
remote: Counting objects: 100% (3252/3252), done.
remote: Compressing objects: 100% (1696/1696), done.
remote: Total 3252 (delta 1716), reused 2407 (delta 1518), pack-reused 0 (from 0)
Receiving objects: 100% (3252/3252), 3.88 MiB | 17.89 MiB/s, done.
Resolving deltas: 100% (1716/1716), done.
-- The C compiler identification is GNU 13.3.0
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- OQS T

In [2]:
#!/usr/bin/env python3
"""pq_exfil_model.py - H1 check: does PQC change exfiltration time of an EXISTING key?
Pure arithmetic model + real ML-DSA sizes from liboqs. No attack code, no hardware."""
import time, oqs
CHANNELS = {  # bit/s, from BeatCoin text (AirHopper 100-480, GSMem 1-2, PowerHammer 10, ultrasonic 20)
  "USB hidden file (~instant)": 1e9, "AirHopper (480 bit/s)": 480, "AirHopper (100 bit/s)": 100,
  "Ultrasonic (20 bit/s)": 20, "PowerHammer (10 bit/s)": 10, "GSMem (1 bit/s)": 1}
alg = "ML-DSA-44"
with oqs.Signature(alg) as s:
    t = time.perf_counter(); pk = s.generate_keypair(); sk = s.export_secret_key(); kg = time.perf_counter()-t
    msg = b"unsigned-tx-demo"; t = time.perf_counter(); sig = s.sign(msg); st = time.perf_counter()-t
    t = time.perf_counter(); ok = s.verify(msg, sig, pk); vt = time.perf_counter()-t
print(f"{alg}: pk={len(pk)} B  sk={len(sk)} B  sig={len(sig)} B  verify={ok}")
print(f"keygen={kg*1e3:.2f} ms  sign={st*1e3:.2f} ms  verify={vt*1e3:.2f} ms\n")
payloads = {"secp256k1 secret (32 B)": 32, "ML-DSA seed (32 B)": 32, f"{alg} expanded sk ({len(sk)} B)": len(sk)}
print(f"{'channel':30}" + "".join(f"{k[:26]:>28}" for k in payloads))
for c, bps in CHANNELS.items():
    print(f"{c:30}" + "".join(f"{b*8/bps:>27.3f}s" for b in payloads.values()))

liboqs-python faulthandler is disabled


INFO:oqs.oqs:liboqs-python faulthandler is disabled


ML-DSA-44: pk=1312 B  sk=2560 B  sig=2420 B  verify=True
keygen=1.03 ms  sign=0.13 ms  verify=0.05 ms

channel                            secp256k1 secret (32 B)          ML-DSA seed (32 B)  ML-DSA-44 expanded sk (256
USB hidden file (~instant)                          0.000s                      0.000s                      0.000s
AirHopper (480 bit/s)                               0.533s                      0.533s                     42.667s
AirHopper (100 bit/s)                               2.560s                      2.560s                    204.800s
Ultrasonic (20 bit/s)                              12.800s                     12.800s                   1024.000s
PowerHammer (10 bit/s)                             25.600s                     25.600s                   2048.000s
GSMem (1 bit/s)                                   256.000s                    256.000s                  20480.000s
